In [2]:
import numpy as np
import os


def rezolva_gauss_seidel(d0, d1, d2, b, p_precizie):
    n = len(d0)
    epsilon = 10 ** (-p_precizie)

    # 1. Determinarea ordinelor p și q conform lungimii vectorilor
    p = n - len(d1)
    q = n - len(d2)

    print(f"Ordin diagonală p = {p}")
    print(f"Ordin diagonală q = {q}")
    # 2. Verificarea elementelor de pe diagonala principală
    if np.any(np.abs(d0) <= epsilon):
        return None, "Eroare: Element nul pe diagonala principală."

    # 3. Metoda Gauss-Seidel
    # Se poate folosi un singur vector pentru actualizarea x(k+1)
    x_gs = np.zeros(n)
    k_max = 10000  #
    k = 0

    while k < k_max:
        x_vechi = x_gs.copy()

        for i in range(n):
            suma = 0
            # Folosim simetria matricei A
            # Diagonala secundară de ordin p
            if i - p >= 0:
                suma += d1[i - p] * x_gs[i - p]  # Element inferior
            if i + p < n:
                suma += d1[i] * x_gs[i + p]  # Element superior

            # Diagonala secundară de ordin q
            if i - q >= 0:
                suma += d2[i - q] * x_gs[i - q]  # Element inferior
            if i + q < n:
                suma += d2[i] * x_gs[i + q]  # Element superior

            # Actualizarea x_i conform formulei (2)
            x_gs[i] = (b[i] - suma) / d0[i]

        # Calcularea normei pentru convergență
        delta_x = np.linalg.norm(x_gs - x_vechi, ord=np.inf)
        k += 1

        if delta_x < epsilon:
            break
        if delta_x > 10 ** 10:  # Limită divergență
            return None, f"Divergență după {k} iterații."

    # 4. Calculul y = Ax_GS (o singură parcurgere)
    y = np.zeros(n)
    for i in range(n):
        val_y = d0[i] * x_gs[i]
        if i - p >= 0: val_y += d1[i - p] * x_gs[i - p]
        if i + p < n: val_y += d1[i] * x_gs[i + p]
        if i - q >= 0: val_y += d2[i - q] * x_gs[i - q]
        if i + q < n: val_y += d2[i] * x_gs[i + q]
        y[i] = val_y

    # 5. Calculul normei erorii ||Ax - b||
    eroare = np.linalg.norm(y - b, ord=np.inf)
    return x_gs, eroare


def ruleaza_testele():
    # Definim soluțiile teoretice pentru validare
    solutii_asteptate = {
        1: lambda i, n: 2.0 / 3.0,
        2: lambda i, n: i * (4.0 / 7.0),
        3: lambda i, n: 2.0,
        4: lambda i, n: float(i),
        5: lambda i, n: i * 0.4
    }

    for i in range(1, 6):
        print(f"\n--- Analiză Sistemul {i} ---")
        try:
            d0 = np.loadtxt(f'd0_{i}.txt')
            d1 = np.loadtxt(f'd1_{i}.txt')
            d2 = np.loadtxt(f'd2_{i}.txt')
            b = np.loadtxt(f'b_{i}.txt')

            x_final, err = rezolva_gauss_seidel(d0, d1, d2, b, p_precizie=6)

            if x_final is not None:
                # Validare contra soluției din document
                x_teoretic = np.array([solutii_asteptate[i](idx, len(d0)) for idx in range(len(d0))])
                dif = np.linalg.norm(x_final - x_teoretic, ord=np.inf)

                print(f"Dimensiune n: {len(d0)}")
                print(f"Eroare calculată ||Ax-b||: {err:.2e}")
                print(f"Diferență față de soluția exactă: {dif:.2e}")
            else:
                print(f"Sistemul {i} nu a putut fi rezolvat: {err}")
        except FileNotFoundError:
            print(f"Fișierele pentru sistemul {i} nu au fost găsite.")


if __name__ == "__main__":
    ruleaza_testele()


--- Analiză Sistemul 1 ---
Ordin diagonală p = 1
Ordin diagonală q = 2026
Dimensiune n: 12026
Eroare calculată ||Ax-b||: 3.05e-06
Diferență față de soluția exactă: 5.26e-08

--- Analiză Sistemul 2 ---
Ordin diagonală p = 2
Ordin diagonală q = 2026
Dimensiune n: 22026
Eroare calculată ||Ax-b||: 4.24e-06
Diferență față de soluția exactă: 7.16e-08

--- Analiză Sistemul 3 ---
Ordin diagonală p = 8
Ordin diagonală q = 2026
Dimensiune n: 82026
Eroare calculată ||Ax-b||: 2.37e-06
Diferență față de soluția exactă: 3.70e-08

--- Analiză Sistemul 4 ---
Ordin diagonală p = 1
Ordin diagonală q = 2
Dimensiune n: 92026
Eroare calculată ||Ax-b||: 6.43e-06
Diferență față de soluția exactă: 9.93e-08

--- Analiză Sistemul 5 ---
Ordin diagonală p = 1
Ordin diagonală q = 2
Sistemul 5 nu a putut fi rezolvat: Divergență după 3 iterații.
